# Typed Composition Search — EDA

Exploratory analysis of benchmark results across 4 models × 4 domains × 6 strategies.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Load Results

In [ ]:
results_dir = Path("../benchmarks/results")

rows = []
query_rows = []
for f in sorted(results_dir.glob("*.json")):
    data = json.loads(f.read_text())
    meta = data["meta"]
    for s in data["strategies"]:
        row = {
            "model": meta["model"],
            "domain": meta["domain"],
            "strategy": s.get("strategy_key", s.get("strategy", "?")),
            "precision": s["precision"],
            "recall": s["recall"],
            "f1": s["f1"],
            "hallucinated": s.get("hallucinated", 0),
            "avg_tools": s.get("avg_tools", 0),
            "total_tools": s.get("total_tools", 0),
            "pruning": s.get("pruning", 0),
            "latency_avg": s.get("latency_avg", 0),
            "avg_prompt_tokens": s.get("avg_prompt_tokens", 0),
            "n": s.get("n", 0),
        }
        rows.append(row)
        for q in s.get("per_query", []):
            query_rows.append({
                "model": meta["model"],
                "domain": meta["domain"],
                "strategy": s.get("strategy_key", s.get("strategy", "?")),
                **q,
            })

df = pd.DataFrame(rows)
qdf = pd.DataFrame(query_rows)
print(f"Strategies: {len(df)} rows, Queries: {len(qdf)} rows")
df.head()

## 2. Strategy Overview — F1 by Model × Domain

In [ ]:
CORE_STRATEGIES = ["baseline", "retrieval", "graph", "graph-reverse-probs", "oracle-graph", "model-types"]
core = df[df["strategy"].isin(CORE_STRATEGIES)].copy()

pivot = core.pivot_table(index=["model", "domain"], columns="strategy", values="f1")
pivot = pivot.reindex(columns=[s for s in CORE_STRATEGIES if s in pivot.columns])
pivot.style.background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1).format("{:.2f}")

## 3. F1 Heatmap

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
models = sorted(core["model"].unique())

for ax, model in zip(axes, models):
    sub = core[core["model"] == model]
    piv = sub.pivot_table(index="domain", columns="strategy", values="f1")
    piv = piv.reindex(columns=[s for s in CORE_STRATEGIES if s in piv.columns])
    sns.heatmap(piv, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
    ax.set_title(model)
    ax.set_ylabel("" if ax != axes[0] else "domain")

fig.suptitle("F1 by Strategy × Domain", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Hallucinations

In [ ]:
hal = core.pivot_table(index=["model", "domain"], columns="strategy", values="hallucinated")
hal = hal.reindex(columns=[s for s in CORE_STRATEGIES if s in hal.columns])
hal.style.background_gradient(cmap="RdYlGn_r", axis=None).format("{:.0f}")

## 5. Token Efficiency

In [ ]:
tok = core.pivot_table(index=["model", "domain"], columns="strategy", values="avg_prompt_tokens")
tok = tok.reindex(columns=[s for s in CORE_STRATEGIES if s in tok.columns])
tok.style.background_gradient(cmap="RdYlGn_r", axis=None).format("{:.0f}")

## 6. Graph-Forward vs Graph-Reverse-Probs (Planning Direction)

In [ ]:
direction = core[core["strategy"].isin(["graph", "graph-reverse-probs"])].copy()
piv = direction.pivot_table(index=["model", "domain"], columns="strategy", values=["f1", "precision", "recall"])
piv

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fwd = direction[direction["strategy"] == "graph"].set_index(["model", "domain"])["f1"]
rev = direction[direction["strategy"] == "graph-reverse-probs"].set_index(["model", "domain"])["f1"]
comp = pd.DataFrame({"graph-forward": fwd, "graph-reverse-probs": rev}).dropna()
comp.plot.bar(ax=ax, rot=45)
ax.set_ylabel("F1")
ax.set_title("Forward vs Reverse Planning Direction")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Decomposition: Type Prediction × Graph Reachability

In [ ]:
oracle = core[core["strategy"] == "oracle-graph"].set_index(["model", "domain"])
types = core[core["strategy"] == "model-types"].set_index(["model", "domain"])
best = core[core["strategy"] == "graph-reverse-probs"].set_index(["model", "domain"])

decomp = pd.DataFrame({
    "oracle_f1": oracle["f1"],
    "type_recall": types["recall"] if "recall" in types.columns else None,
    "e2e_f1": best["f1"],
}).dropna()
decomp

## 8. Per-Query Error Analysis

In [ ]:
errors = qdf[(qdf["strategy"] == "graph-reverse-probs") & (qdf["f1"] < 1.0)].copy()
if len(errors) > 0:
    print(f"{len(errors)} queries with F1 < 1.0")
    error_cols = ["model", "domain", "id", "category", "expected_source", "expected_target",
                  "predicted_source", "predicted_target", "path_found", "f1"]
    display(errors[[c for c in error_cols if c in errors.columns]].sort_values("f1"))
else:
    print("No errors found")

## 9. Performance by Query Category

In [ ]:
cat_df = qdf[qdf["strategy"].isin(["baseline", "graph", "graph-reverse-probs"])].copy()
if "category" in cat_df.columns:
    cat_f1 = cat_df.groupby(["strategy", "category"])["f1"].mean().unstack("strategy")
    cat_f1 = cat_f1.reindex(columns=["baseline", "graph", "graph-reverse-probs"])
    cat_f1.plot.bar(figsize=(10, 5), rot=0)
    plt.ylabel("Mean F1")
    plt.title("F1 by Query Category")
    plt.ylim(0, 1)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("No category column found")

## 10. Cross-Model Variance

In [ ]:
variance = core[core["strategy"].isin(["baseline", "graph-reverse-probs"])].copy()
var_stats = variance.groupby(["domain", "strategy"])["f1"].agg(["mean", "std"]).unstack("strategy")
var_stats